### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="obesity_estimation",
    dataset_year="2019",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5H31Z",
    download_description="""
wget https://archive.ics.uci.edu/static/public/544/estimation+of+obesity+levels+based+on+eating+habits+and+physical+condition.zip && unzip estimation+of+obesity+levels+based+on+eating+habits+and+physical+condition.zip && rm estimation+of+obesity+levels+based+on+eating+habits+and+physical+condition.zip
mkdir -p local-data-warehouse/obesity_estimation && mv *.csv local-data-warehouse/obesity_estimation/
""",
    # References
    academic_reference_bibtex=r"""@article{palechor2019dataset,
    title={Dataset for estimation of obesity levels based on eating habits and physical condition in individuals from Colombia, Peru and Mexico},
    author={Palechor, Fabio Mendoza and De la Hoz Manotas, Alexis},
    journal={Data in brief},
    volume={25},
    pages={104344},
    year={2019},
    publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="palechor2019dataset",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We use only the real data, throwing away the SMOTE samples.
- We make the task a regression task, re-creating the body mass values underlying the discretized obesity levels, and drop weight and height which leak the target partially. Thus, we create a regression task of estimating the body mass of individuals based on their eating habits and other questions from the questionnaire. The original task is trivial to solve due to the leak and thus not interesting.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="BodyMass",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "ObesityDataSet_raw_and_data_sinthetic.csv")

# Filter to only real data (real data is int and not float values, see original .csv file)
df = df.iloc[:498]

# Creat real target
df["BodyMass"] = df["Weight"] / (df["Height"] ** 2)
df = df.drop(columns=["Height", "Weight", "NObeyesdad"])

as_cat_type = [
    "Gender", "family_history_with_overweight", "FAVC", "FCVC",
    "CAEC", "SMOKE", "SCC", "CALC", "MTRANS",
    # Numerical on UCI, but from the paper we know this was categorial in the questionnaire:
    "CH2O", "FAF", "TUE", "NCP",
]
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 498
Columns: 15
Use sampling: False (sample size: 498)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['Age', 'MTRANS', 'FAF', 'CAEC', 'CALC', 'FCVC', 'NCP', 'CH2O', 'TUE', 'FAVC']
Rows remaining as candidates after top-10 filter: 42 (of 498)

#### Duplicate Report
Total duplicate rows: 10 (2.01% of dataset)
Duplicate rows ignoring target: 14 (2.81% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,Gender,Age,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,BodyMass
0,Male,20.0,yes,no,2.0,3.0,Sometimes,no,3.0,no,3.0,0.0,no,Motorbike,30.718336
1,Male,19.0,yes,no,2.0,3.0,Sometimes,no,3.0,no,2.0,1.0,Sometimes,Bike,18.991965
2,Female,21.0,yes,yes,3.0,1.0,Sometimes,yes,3.0,no,0.0,0.0,Sometimes,Public_Transportation,24.840980
3,Female,38.0,yes,yes,2.0,3.0,Sometimes,no,2.0,no,2.0,0.0,no,Public_Transportation,22.233789
4,Female,19.0,yes,yes,3.0,3.0,Sometimes,no,1.0,no,1.0,1.0,no,Public_Transportation,19.705532


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Gender,category,0.0,0.0,2.0,"Male, Female"
1,family_history_with_overweight,category,0.0,0.0,2.0,"yes, no"
2,FAVC,category,0.0,0.0,2.0,"yes, no"
3,FCVC,category,0.0,0.0,3.0,"2.0, 3.0, 1.0"
4,NCP,category,0.0,0.0,3.0,"3.0, 1.0, 4.0"
5,CAEC,category,0.0,0.0,4.0,"Sometimes, Frequently, Always, no"
6,SMOKE,category,0.0,0.0,2.0,"no, yes"
7,CH2O,category,0.0,0.0,3.0,"2.0, 1.0, 3.0"
8,SCC,category,0.0,0.0,2.0,"no, yes"
9,FAF,category,0.0,0.0,4.0,"0.0, 1.0, 2.0, 3.0"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Age,498.0,23.146586,6.721583,14.000000,61.00000
BodyMass,498.0,24.313530,4.772267,13.291588,49.47239


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column                         rank                                     
CAEC                           1                 Sometimes    289  58.03
                               2                Frequently    136  27.31
                               3                    Always     53  10.64
                               4                        no     20   4.02
CALC                           1                 Sometimes    273  54.82
                               2                        no    179  35.94
                               3                Frequently     45   9.04
                               4                    Always      1   0.20
CH2O                           1                       2.0    266  53.41
                               2                       1.0    135  27.11
                               3                       3.0     97  19.48
FAF                            1                       0.0    162  32.53
                               2                       1.0    158  31.73
                               3                       2.0    113  22.69
                               4                       3.0     65  13.05
FAVC                           1                       yes    348  69.88
                               2                        no    150  30.12
FCVC                           1                       2.0    272  54.62
                               2                       3.0    194  38.96
                               3                       1.0     32   6.43
Gender                         1                      Male    271  54.42
                               2                    Female    227  45.58
MTRANS                         1     Public_Transportation    326  65.46
                               2                Automobile     99  19.88
                               3                   Walking     55  11.04
                               4                 Motorbike     11   2.21
                               5                      Bike      7   1.41
NCP                            1                       3.0    344  69.08
                               2                       1.0    108  21.69
                               3                       4.0     46   9.24
SCC                            1                        no    443  88.96
                               2                       yes     55  11.04
SMOKE                          1                        no    466  93.57
                               2                       yes     32   6.43
TUE                            1                       0.0    243  48.80
                               2                       1.0    181  36.35
                               3                       2.0     74  14.86
family_history_with_overweight 1                       yes    300  60.24
                               2                        no    198  39.76

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,1.037,0.394,22.775,0.035,log,4589.4,4341.1,lognormal


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to obesity_estimation/019d633a-b275-78cc-8a11-43693dff428b
019d633a-b275-78cc-8a11-43693dff428b
492c6c2ad996cc1f64d832910afd77bf83ef0de5bb550cae93cf7a3e4f99f937
